In [7]:
# Task 1: Imports
## Phase 1 · Foundation & Exploration
# Setup, API key, smoke tests — V3 and R1 response shapes confirmed.
import os
import json
import requests
from dotenv import load_dotenv

load_dotenv()

# Task 2: API key and endpoint
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
API_URL = "https://openrouter.ai/api/v1/chat/completions"

# assert OPENROUTER_API_KEY, "Key not loaded — check your .env file"

HEADERS = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json",
}

payload = {
    "model": "deepseek/deepseek-r1",   # or :free variant if you prefer
    "messages": [
        {"role": "user", "content": "Solve this step by step: If x^2 - 5x + 6 = 0, what are x?"}
    ],
    "max_tokens": 1000,   # generous; thinking tokens count against this
}


resp = requests.post(API_URL, headers=HEADERS, json=payload)
resp.raise_for_status()
data = resp.json()

msg = data["choices"][0]["message"]
print("CONTENT:", msg["content"])
print("REASONING:", (msg.get("reasoning") or "")[:500])   # first 500 chars
print("USAGE:", data["usage"]["completion_tokens_details"])

CONTENT: None
REASONING: Okay, so I need to solve the quadratic equation x squared minus five x plus six equals zero. Hmm, let me think. Quadratic equations usually have two solutions, right? The standard form is ax² + bx + c = 0, and there are different ways to solve them. Maybe I can factor this one? Let me try.

First, the equation is x² - 5x + 6 = 0. Wait, no, the original equation is x² - 5x + 6 = 0. Yeah, that's correct. So, to factor a quadratic equation, I need to find two numbers that multiply to give the const
USAGE: {'reasoning_tokens': 773, 'image_tokens': 0, 'audio_tokens': 0}


In [9]:
## Phase 2 · Core Helper
# Import the reusable client from src/
from pathlib import Path
import sys

# Support running the notebook from either project root or notebooks/
candidate_roots = [Path.cwd(), Path.cwd().parent]
project_root = next((p for p in candidate_roots if (p / "src").exists()), Path.cwd())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.openrouter_client import query_model, extract_answer

result = query_model(
    model="deepseek/deepseek-chat-v3-0324",
    prompt="Write a reply to this review: 'The product broke after one day.'",
    system="You are a polite customer support agent.",
)
print(extract_answer(result))

[Novita · deepseek/deepseek-chat-v3-0324]
  tokens → prompt: 26 | completion: 169 | reasoning: 0 | cost: $0.00019630
Certainly! Here's a polite and professional response to the review:  

---  

**Response:**  

We sincerely apologize for the inconvenience you've experienced with our product. This is certainly not the level of quality we aim to provide, and we’d like to make it right for you.  

Please reach out to our customer support team at [support email/phone] or reply directly here, and we’ll assist you with a replacement or refund right away. Your satisfaction is very important to us, and we appreciate your feedback as it helps us improve.  

Thank you for giving us the opportunity to resolve this for you.  

Best regards,  
[Your Name/Company Name]  

---  

This response acknowledges the issue, offers a solution, and maintains a courteous tone. Let me know if you'd like any adjustments!


In [ ]:
## Task 3 · Email Generation
# Prompt iteration across 3 rounds: naive -> system-constrained -> JSON-structured
from pathlib import Path
import sys
import json
import ast

# Support running the notebook from either project root or notebooks/
candidate_roots = [Path.cwd(), Path.cwd().parent]
project_root = next((p for p in candidate_roots if (p / "src").exists()), Path.cwd())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.openrouter_client import query_model, extract_answer

reviews_path = project_root / "data" / "sample_reviews.txt"
text = reviews_path.read_text(encoding="utf-8")
reviews = ast.literal_eval(text.split("=", 1)[1].strip())

# Round 1: naive, no system prompt, no format control
def generate_email_v1(review: dict) -> str:
    prompt = f"Write a customer support email reply to this review: '{review['text']}'"

    result = query_model(
        model="deepseek/deepseek-chat-v3-0324",
        prompt=prompt,
    )
    return extract_answer(result)

print("=== ROUND 1 ===")
for review in reviews:
    print(f"\n--- Review {review['id']} (rating: {review['rating']}) ---")
    print(generate_email_v1(review))
    print()

SUPPORT_SYSTEM_PROMPT = """You are a customer support specialist for an e-commerce company.
Write professional, empathetic email replies to customer reviews.

Rules:
- Start directly with the email body. No preamble like 'Here is a reply...'
- No placeholders like [Your Name] or [support email]. Use 'our support team' instead.
- Sign off as: The Support Team
- Match tone to sentiment: apologetic for negative, warm for positive, balanced for mixed
- Maximum 3 short paragraphs
- No markdown formatting, no dividers"""

def generate_email_v2(review: dict) -> str:
    prompt = f"""Customer review (rating: {review['rating']}/5):
\"{review['text']}\"

Write the email reply."""

    result = query_model(
        model="deepseek/deepseek-chat-v3-0324",
        prompt=prompt,
        system=SUPPORT_SYSTEM_PROMPT,
    )
    return extract_answer(result)

print("=== ROUND 2 ===")
for review in reviews:
    print(f"\n--- Review {review['id']} (rating: {review['rating']}) ---")
    print(generate_email_v2(review))
    print()

JSON_SYSTEM_PROMPT = """You are a customer support specialist for an e-commerce company.
Reply ONLY with a valid JSON object. No preamble, no markdown, no backticks.
- Never invent contact details, URLs, email addresses, or order numbers
- If you feel compelled to generate another review, stop and return only the JSON object

JSON shape:
{
  "subject": "email subject line",
  "body": "full email body, no placeholders, signed off as The Support Team",
  "sentiment": "positive | negative | mixed",
  "priority": "high | medium | low"
}

Priority rules: rating 1-2 = high, rating 3 = medium, rating 4-5 = low"""

def generate_email_v3(review: dict) -> dict:
    prompt = f"""Customer review (rating: {review['rating']}/5):
\"{review['text']}\"

Reply with the JSON object."""

    result = query_model(
        model="deepseek/deepseek-chat-v3-0324",
        prompt=prompt,
        system=JSON_SYSTEM_PROMPT,
    )

    raw = extract_answer(result)
    # Strip markdown fences if the model adds them anyway
    clean = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(clean)

print("=== ROUND 3 ===")
for review in reviews:
    print(f"\n--- Review {review['id']} (rating: {review['rating']}) ---")
    output = generate_email_v3(review)
    print(json.dumps(output, indent=2))
    print()

=== ROUND 1 ===

--- Review r1 (rating: 1) ---
[Novita · deepseek/deepseek-chat-v3-0324]
  tokens → prompt: 33 | completion: 299 | reasoning: 0 | cost: $0.00034379
Here’s a professional and empathetic customer support email reply you could use:  

---  

**Subject:** We’re Sorry to Hear About Your Experience  

Dear [Customer's Name],  

Thank you for sharing your feedback with us. We’re truly sorry to hear that your product didn’t meet expectations and failed so quickly—that’s certainly not the experience we want for our customers.  

We stand behind the quality of our products, and we’d like to make this right for you. Could you please share your order number and any additional details about the issue? We’d be happy to replace the product or process a refund, whichever you prefer.  

Your satisfaction is important to us, and we’d appreciate the opportunity to regain your trust. Please reply to this email or contact our support team directly at [support email/phone], and we’ll resolve

In [ ]:
# Token cost comparison across rounds
print("Token usage comparison (review r1 only):\n")

for label, fn in [("v1 naive", generate_email_v1), 
                   ("v2 system prompt", generate_email_v2)]:
    result = query_model(
        "deepseek/deepseek-chat-v3-0324",
        f"Write a customer support email reply to this review: '{reviews[0]['text']}'",
        verbose=True
    )
    print(f"{label}: {result['usage']['completion_tokens']} completion tokens\n")

In [ ]:
# Task 3 summary - key learnings
observations = {
    "round_1": "Naive prompt: preamble, placeholders, meta-commentary. Avg ~267 tokens.",
    "round_2": "System prompt: cleaner output but prompt injection caused extra review hallucination on r2/r3.",
    "round_3": "JSON output: clean structure, correct classification, ~138 avg tokens. One hallucinated email address.",
    "prompt_engineering_lessons": [
        "System prompts cut completion tokens significantly on targeted tasks",
        "Template structure can cause model to hallucinate continuation patterns",
        "JSON output enforces format discipline and enables downstream consumption",
        "Models may invent specifics (emails, order numbers) unless explicitly prohibited"
    ]
}

import json
print(json.dumps(observations, indent=2))

## Task 4 · Code Generation with DeepSeek R1

In [14]:
import re

CODE_SYSTEM_PROMPT = """You are an expert Python engineer.
When given a coding problem:
- Write clean, correct Python code
- Include the function implementation only
- Add a brief docstring
- Do not include example usage or test calls outside the function
- Do not explain the code outside of the function"""

In [15]:
def extract_code(text: str) -> str:
    """Pull code block out of model response."""
    text = (text or "").strip()
    match = re.search(r"```(?:python)?\n(.*?)```", text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return text.strip()


def generate_code(problem: dict) -> dict:
    result = query_model(
        model="deepseek/deepseek-r1",
        prompt=problem["description"],
        system=CODE_SYSTEM_PROMPT,
        max_tokens=2000,
        include_reasoning=True,
    )

    raw = extract_answer(result)
    code = extract_code(raw)

    return {
        "id": problem["id"],
        "code": code,
        "reasoning_tokens": result["usage"].get(
            "completion_tokens_details", {}
        ).get("reasoning_tokens", 0),
        "completion_tokens": result["usage"].get("completion_tokens", 0),
        "cost": result["usage"].get("cost", 0),
    }


def execute_code(code: str, test_cases: list) -> dict:
    """Run generated code against test cases and return pass/fail details."""
    results = []
    namespace = {}

    try:
        exec(code, namespace)
    except Exception as e:
        return {"success": False, "error": f"Syntax/compile error: {e}", "tests": []}

    for test in test_cases:
        try:
            actual = eval(test["call"], namespace)
            passed = actual == test["expected"]
            results.append({
                "call": test["call"],
                "expected": test["expected"],
                "actual": actual,
                "passed": passed,
            })
        except Exception as e:
            results.append({
                "call": test["call"],
                "expected": test["expected"],
                "actual": None,
                "passed": False,
                "error": str(e),
            })

    return {"success": all(r["passed"] for r in results), "tests": results}


def fix_code(problem: dict, broken_code: str, failures: list) -> dict:
    """Send failed test cases back to R1 for correction."""
    failure_summary = "\n".join([
        f"- {f['call']} -> expected {f['expected']}, got {f.get('actual')} "
        f"{'ERROR: ' + f['error'] if 'error' in f else ''}"
        for f in failures
        if not f["passed"]
    ])

    prompt = f"""This Python code has failing tests:
```python
{broken_code}
```

Failing test cases:
{failure_summary}

Fix the code so all tests pass. Return only the corrected function."""

    result = query_model(
        model="deepseek/deepseek-r1",
        prompt=prompt,
        system=CODE_SYSTEM_PROMPT,
        max_tokens=2000,
        include_reasoning=True,
    )

    raw = extract_answer(result)
    code = extract_code(raw)

    return {
        "id": problem["id"],
        "code": code,
        "reasoning_tokens": result["usage"].get(
            "completion_tokens_details", {}
        ).get("reasoning_tokens", 0),
        "cost": result["usage"].get("cost", 0),
        "is_fix": True,
    }

In [16]:
coding_problems = [
    {
        "id": "p1",
        "description": "Write a function flatten(lst) that flattens arbitrarily nested lists into a single list while preserving order.",
    },
    {
        "id": "p2",
        "description": "Write a function caesar_cipher(text, shift) that shifts alphabetic characters by shift positions, preserving case and non-letters.",
    },
    {
        "id": "p3",
        "description": "Write a function find_duplicates(nums) that returns sorted unique duplicated values from a list of integers.",
    },
]

print(f"Loaded {len(coding_problems)} coding problems.")

Loaded 3 coding problems.


In [18]:
# Cell 5 - Round 1: generate
print("=== ROUND 1: Code Generation ===\n")
generated = []
for problem in coding_problems:
    print(f"--- {problem['id']} ---")
    output = generate_code(problem)
    generated.append(output)
    print(output["code"])
    print(f"\nReasoning tokens: {output['reasoning_tokens']} | "
          f"Total tokens: {output['completion_tokens']} | "
          f"Cost: ${output['cost']:.8f}\n")

=== ROUND 1: Code Generation ===

--- p1 ---
[Novita · deepseek/deepseek-r1]
  tokens → prompt: 84 | completion: 2701 | reasoning: 2094 | cost: $0.00681130
def flatten(lst):
    """Flattens an arbitrarily nested list structure into a single list while preserving order.
    
    Args:
        lst (list): The nested list to flatten.
    
    Returns:
        list: The flattened list.
        
    Example:
        >>> flatten([1, [2, [3, 4], 5]])
        [1, 2, 3, 4, 5]
    """
    if not lst:
        return []

    stack = []
    result = []
    it = iter(lst)
    
    while True:
        try:
            element = next(it)
            if isinstance(element, list):
                stack.append(it)
                it = iter(element)
            else:
                result.append(element)
        except StopIteration:
            if stack:
                it = stack.pop()
            else:
                break
                
    return result

Reasoning tokens: 2094 | Total tokens: 270

In [19]:
# Cell 6 - Round 2: execute and validate
test_suites = {
    "p1": [
        {"call": "flatten([1, [2, [3, 4]], 5])",       "expected": [1, 2, 3, 4, 5]},
        {"call": "flatten([[1, 2], [3, [4, 5]]])",     "expected": [1, 2, 3, 4, 5]},
        {"call": "flatten([1, 2, 3])",                  "expected": [1, 2, 3]},
        {"call": "flatten([])",                         "expected": []},
    ],
    "p2": [
        {"call": "caesar_cipher('Hello, World!', 3)",  "expected": "Khoor, Zruog!"},
        {"call": "caesar_cipher('xyz', 3)",            "expected": "abc"},
        {"call": "caesar_cipher('ABC', 1)",            "expected": "BCD"},
        {"call": "caesar_cipher('Hello!', 0)",         "expected": "Hello!"},
    ],
    "p3": [
        {"call": "find_duplicates([1, 2, 3, 2, 4, 3, 5])", "expected": [2, 3]},
        {"call": "find_duplicates([1, 1, 1])",              "expected": [1]},
        {"call": "find_duplicates([1, 2, 3])",              "expected": []},
        {"call": "find_duplicates([])",                     "expected": []},
    ],
}

print("=== ROUND 2: Execution & Validation ===\n")
validation_results = []
for output in generated:
    tests = test_suites[output["id"]]
    result = execute_code(output["code"], tests)
    validation_results.append({**output, "validation": result})

    status = "PASS" if result["success"] else "FAIL"
    print(f"--- {output['id']} {status} ---")
    for t in result["tests"]:
        icon = "PASS" if t["passed"] else "FAIL"
        print(f"  {icon} {t['call']}")
        if not t["passed"]:
            print(f"    expected: {t['expected']}")
            print(f"    actual:   {t.get('actual')}")
            if "error" in t:
                print(f"    error:    {t['error']}")
    print()

=== ROUND 2: Execution & Validation ===

--- p1 PASS ---
  PASS flatten([1, [2, [3, 4]], 5])
  PASS flatten([[1, 2], [3, [4, 5]]])
  PASS flatten([1, 2, 3])
  PASS flatten([])

--- p2 PASS ---
  PASS caesar_cipher('Hello, World!', 3)
  PASS caesar_cipher('xyz', 3)
  PASS caesar_cipher('ABC', 1)
  PASS caesar_cipher('Hello!', 0)

--- p3 PASS ---
  PASS find_duplicates([1, 2, 3, 2, 4, 3, 5])
  PASS find_duplicates([1, 1, 1])
  PASS find_duplicates([1, 2, 3])
  PASS find_duplicates([])



In [20]:
# Cell 7 - Round 3: feedback loop
print("=== ROUND 3: Feedback Loop ===\n")
final_results = []

for vr in validation_results:
    if vr["validation"]["success"]:
        print(f"{vr['id']}: already passing, no fix needed PASS")
        final_results.append(vr)
        continue

    failures = vr["validation"]["tests"]
    problem = next(p for p in coding_problems if p["id"] == vr["id"])

    print(f"{vr['id']}: fixing {sum(1 for f in failures if not f['passed'])} "
          f"failing test(s)...")

    fixed = fix_code(problem, vr["code"], failures)
    retest = execute_code(fixed["code"], test_suites[vr["id"]])

    status = "FIXED" if retest["success"] else "STILL FAILING"
    print(f"  Result: {status}")
    print(f"  Fix cost: ${fixed['cost']:.8f}\n")

    final_results.append({**fixed, "validation": retest})

=== ROUND 3: Feedback Loop ===

p1: already passing, no fix needed PASS
p2: already passing, no fix needed PASS
p3: already passing, no fix needed PASS


In [21]:
# Cell 8 - summary
print("=== TASK 4 SUMMARY ===\n")
print(f"{'Problem':<10} {'Status':<12} {'R-Tokens':<12} {'Cost':<14} {'Fixed?'}")
print("-" * 60)

total_cost = 0
for r in final_results:
    status = "PASS" if r["validation"]["success"] else "FAIL"
    fixed = "yes" if r.get("is_fix") else "no"
    cost = r.get("cost", 0)
    total_cost += cost
    print(f"{r['id']:<10} {status:<12} "
          f"{r.get('reasoning_tokens', 0):<12} "
          f"${cost:<13.8f} {fixed}")

print("-" * 60)
print(f"{'TOTAL':<10} {'':12} {'':12} ${total_cost:.8f}")

=== TASK 4 SUMMARY ===

Problem    Status       R-Tokens     Cost           Fixed?
------------------------------------------------------------
p1         PASS         2094         $0.00681130    no
p2         PASS         220          $0.00208980    no
p3         PASS         291          $0.00181740    no
------------------------------------------------------------
TOTAL                                $0.01071850


## Task 5 · Text Summarization

In [22]:
# Cell 2 - summarization system prompt and helper
if "paper_excerpt" not in globals():
    paper_path = project_root / "data" / "paper_excerpt.txt"
    paper_excerpt = paper_path.read_text(encoding="utf-8")

def summarize(
    text: str,
    style: str = "paragraph",
    max_words: int = 100,
    audience: str = "general",
    model: str = "deepseek/deepseek-chat-v3-0324",
) -> dict:
    """
    style:    paragraph | bullets | tldr | structured
    audience: general | technical | executive
    """

    style_instructions = {
        "paragraph": f"Write a single flowing paragraph of no more than {max_words} words.",
        "bullets":   f"Write exactly 4 bullet points, each under 20 words.",
        "tldr":      f"Write a single sentence TL;DR of no more than 25 words.",
        "structured": (
            "Write a structured summary with these exact sections:\n"
            "CORE IDEA: one sentence\n"
            "KEY POINTS: three bullet points\n"
            "IMPLICATIONS: one sentence"
        ),
    }

    audience_instructions = {
        "general":   "Avoid jargon. Explain technical terms if used.",
        "technical": "Assume the reader has a CS background. Use precise terminology.",
        "executive": "Focus on business and strategic implications. No implementation details.",
    }

    system = f"""You are an expert technical writer specializing in AI research.
Summarize the provided text according to these rules:
- {style_instructions[style]}
- Audience: {audience_instructions[audience]}
- Do not include preamble like 'Here is a summary...'
- Do not refer to 'the text' or 'the passage' — write directly"""

    result = query_model(
        model=model,
        prompt=f"Summarize this:\n\n{text}",
        system=system,
        max_tokens=500,
    )

    return {
        "style": style,
        "audience": audience,
        "content": extract_answer(result),
        "tokens": result["usage"].get("completion_tokens"),
        "cost": result["usage"].get("cost", 0),
    }

In [23]:
# Cell 3 - Round 1: style variations (same audience, four formats)
print("=== ROUND 1: Style Variations (technical audience) ===\n")

styles = ["tldr", "bullets", "paragraph", "structured"]
style_results = []

for style in styles:
    r = summarize(paper_excerpt, style=style, audience="technical")
    style_results.append(r)
    print(f"--- {style.upper()} ({r['tokens']} tokens) ---")
    print(r["content"])
    print()

=== ROUND 1: Style Variations (technical audience) ===

[GMICloud · deepseek/deepseek-chat-v3-0324]
  tokens → prompt: 465 | completion: 37 | reasoning: 0 | cost: $0.00017703
--- TLDR (37 tokens) ---
TL;DR: Transformers revolutionized NLP via parallel self-attention, multi-head mechanisms, and positional encodings, enabling models like BERT and GPT with emergent capabilities at scale.

[GMICloud · deepseek/deepseek-chat-v3-0324]
  tokens → prompt: 464 | completion: 103 | reasoning: 0 | cost: $0.00025198
--- BULLETS (103 tokens) ---
- Transformers replaced RNNs with parallel self-attention, capturing long-range dependencies via token relevance weighting.  
- Multi-head attention enables parallel focus on different input aspects (syntax, semantics), concatenated for final representation.  
- Positional encodings (sinusoidal/learned) inject order information since transformers lack inherent sequential processing.  
- Encoder-only (BERT), decoder-only (GPT), and encoder-decoder (T5) models

In [24]:
# Cell 4 - Round 2: audience variations (same format, three audiences)
print("=== ROUND 2: Audience Variations (structured format) ===\n")

audiences = ["general", "technical", "executive"]
audience_results = []

for audience in audiences:
    r = summarize(paper_excerpt, style="structured", audience=audience)
    audience_results.append(r)
    print(f"--- {audience.upper()} ---")
    print(r["content"])
    print()

=== ROUND 2: Audience Variations (structured format) ===

[SiliconFlow · deepseek/deepseek-chat-v3-0324]
  tokens → prompt: 479 | completion: 172 | reasoning: 0 | cost: $0.00029175
--- GENERAL ---
**CORE IDEA:** Transformers revolutionized natural language processing by using parallel processing and self-attention to analyze text more effectively than older sequential models.  

**KEY POINTS:**  
- **Self-attention mechanism:** Instead of processing words one by one, transformers analyze entire sequences at once, evaluating how each word relates to others for better context understanding.  
- **Multi-head attention:** The model runs multiple attention operations in parallel, each focusing on different aspects of language (e.g., grammar or meaning) to create richer representations.  
- **Positional encoding:** Since transformers don’t process words in order, they add positional markers (like numerical tags) to retain information about word positions.  

**IMPLICATIONS:** The efficiency 

In [25]:
# Cell 5 - Round 3: chaining (summarize -> then summarize the summary)
print("=== ROUND 3: Chained Summarization ===\n")

# Step 1: detailed summary
step1 = summarize(paper_excerpt, style="paragraph", max_words=150, audience="technical")
print("Step 1 - detailed summary:")
print(step1["content"])
print(f"({step1['tokens']} tokens)\n")

# Step 2: compress that summary further
step2 = summarize(step1["content"], style="tldr", audience="general")
print("Step 2 - TL;DR of the summary:")
print(step2["content"])
print(f"({step2['tokens']} tokens)\n")

# Step 3: one more - executive framing
step3 = summarize(step2["content"], style="structured", audience="executive")
print("Step 3 - executive structured view:")
print(step3["content"])
print(f"({step3['tokens']} tokens)\n")

total_chain_cost = step1["cost"] + step2["cost"] + step3["cost"]
print(f"Total chain cost: ${total_chain_cost:.8f}")

=== ROUND 3: Chained Summarization ===

[GMICloud · deepseek/deepseek-chat-v3-0324]
  tokens → prompt: 464 | completion: 161 | reasoning: 0 | cost: $0.00031810
Step 1 - detailed summary:
Transformers revolutionized NLP by replacing sequential RNN processing with parallel self-attention, enabling direct modeling of long-range token dependencies. Their key innovation—multi-head attention—parallelizes distinct attention patterns (e.g., syntactic vs semantic) before combining them into unified representations. Positional encodings (sinusoidal or learned) compensate for the architecture's lack of inherent sequence awareness. The framework spawned three dominant variants: encoder-only models (e.g., BERT) for understanding tasks, decoder-only models (e.g., GPT) for generation, and encoder-decoder architectures (e.g., T5) for sequence-to-sequence tasks. Scaling these models revealed emergent capabilities at specific parameter thresholds, driving trends toward larger architectures while raising

In [26]:
# Cell 6 - comparison table
print("=== TASK 5 SUMMARY ===\n")
print(f"{'Style':<12} {'Audience':<12} {'Tokens':<10} {'Cost'}")
print("-" * 48)

for r in style_results + audience_results:
    print(f"{r['style']:<12} {r['audience']:<12} "
          f"{r['tokens']:<10} ${r['cost']:.8f}")

print("-" * 48)
print("\nChain:")
for i, r in enumerate([step1, step2, step3], 1):
    print(f"  Step {i} ({r['style']:<12}): {r['tokens']} tokens  ${r['cost']:.8f}")
print(f"  Chain total:              ${total_chain_cost:.8f}")

=== TASK 5 SUMMARY ===

Style        Audience     Tokens     Cost
------------------------------------------------
tldr         technical    37         $0.00017703
bullets      technical    103        $0.00025198
paragraph    technical    129        $0.00024500
structured   technical    163        $0.00028350
structured   general      172        $0.00029175
structured   technical    182        $0.00033398
structured   executive    127        $0.00022302
------------------------------------------------

Chain:
  Step 1 (paragraph   ): 161 tokens  $0.00031810
  Step 2 (tldr        ): 52 tokens  $0.00012466
  Step 3 (structured  ): 111 tokens  $0.00017149
  Chain total:              $0.00061425


## Task 6 · Instagram Caption Generation

In [27]:
# Image scenarios (since we're not passing actual images)
scenarios = [
    {
        "id": "s1",
        "description": "A golden sunset over the ocean with silhouettes of palm trees",
        "context": "travel lifestyle blog",
    },
    {
        "id": "s2",
        "description": "A flat lay of a MacBook, coffee cup, notebook and succulent plant on a white desk",
        "context": "productivity and work from home",
    },
    {
        "id": "s3",
        "description": "A bowl of colorful ramen with soft boiled egg, nori and sesame seeds",
        "context": "food photography",
    },
]

CAPTION_TONES = {
    "inspirational": (
        "Write in an uplifting, motivational tone. "
        "Use vivid sensory language. End with a reflective question."
    ),
    "witty": (
        "Write in a clever, slightly humorous tone. "
        "Include a subtle pun or wordplay if natural. Keep it light."
    ),
    "minimal": (
        "Write in a clean, understated tone. "
        "No more than 2 sentences. Let the image speak."
    ),
    "storytelling": (
        "Write as if narrating a moment in time. "
        "First person. Draw the reader into the scene."
    ),
}

def generate_caption(
    scenario: dict,
    tone: str = "inspirational",
    num_hashtags: int = 5,
    model: str = "deepseek/deepseek-chat-v3-0324",
) -> dict:

    system = f"""You are a professional Instagram content creator.
Generate engaging captions for Instagram posts.

Rules:
- Tone: {CAPTION_TONES[tone]}
- Add exactly {num_hashtags} relevant hashtags at the end on a new line
- No preamble like 'Here is a caption...'
- No quotation marks around the caption
- Context: {scenario['context']}"""

    prompt = f"Write an Instagram caption for this image: {scenario['description']}"

    result = query_model(
        model=model,
        prompt=prompt,
        system=system,
        max_tokens=300,
    )

    content = extract_answer(result)

    # Split caption body from hashtags
    parts = content.strip().rsplit("\n", 1)
    body = parts[0].strip() if len(parts) > 1 else content.strip()
    hashtags = parts[1].strip() if len(parts) > 1 else ""

    return {
        "id": scenario["id"],
        "tone": tone,
        "body": body,
        "hashtags": hashtags,
        "full": content,
        "tokens": result["usage"].get("completion_tokens"),
        "cost": result["usage"].get("cost", 0),
    }

In [28]:
print("=== ROUND 1: Tone Variations (sunset scenario) ===\n")

tone_results = []
sunset = scenarios[0]

for tone in CAPTION_TONES:
    r = generate_caption(sunset, tone=tone)
    tone_results.append(r)
    print(f"--- {tone.upper()} ({r['tokens']} tokens) ---")
    print(r["body"])
    print(r["hashtags"])
    print()

=== ROUND 1: Tone Variations (sunset scenario) ===

[Novita · deepseek/deepseek-chat-v3-0324]
  tokens → prompt: 104 | completion: 77 | reasoning: 0 | cost: $0.00011432
--- INSPIRATIONAL (77 tokens) ---
Golden hour magic 🌅✨ The sky paints itself in hues of amber and rose, while the ocean whispers secrets to the shore. Silhouettes of swaying palms dance against the fading light—nature’s perfect symphony. Where would you chase this kind of sunset?
#TravelVibes #SunsetLover #BeachLife #Wanderlust #GoldenHour

[SiliconFlow · deepseek/deepseek-chat-v3-0324]
  tokens → prompt: 108 | completion: 38 | reasoning: 0 | cost: $0.00006500
--- WITTY (38 tokens) ---
Golden hour just got an upgrade – and the palm trees are loving the spotlight.
#SunsetVibes #BeachLife #GoldenHourMagic #Wanderlust #OceanViews

[SiliconFlow · deepseek/deepseek-chat-v3-0324]
  tokens → prompt: 106 | completion: 28 | reasoning: 0 | cost: $0.00005450
--- MINIMAL (28 tokens) ---
Golden hour magic washed over the horizon lik

In [29]:
print("=== ROUND 2: All Scenarios (storytelling tone) ===\n")

scenario_results = []

for scenario in scenarios:
    r = generate_caption(scenario, tone="storytelling", num_hashtags=7)
    scenario_results.append(r)
    print(f"--- {scenario['id'].upper()}: {scenario['context']} ---")
    print(r["body"])
    print(r["hashtags"])
    print()

=== ROUND 2: All Scenarios (storytelling tone) ===

[GMICloud · deepseek/deepseek-chat-v3-0324]
  tokens → prompt: 105 | completion: 77 | reasoning: 0 | cost: $0.00011823
--- S1: travel lifestyle blog ---
The sky melts into liquid gold, spilling over the horizon while palm trees sway like shadows against the fire. Time slows here—just the waves, the breeze, and this endless glow. Some moments are too beautiful to keep to yourself.
#SunsetMagic #OceanVibes #GoldenHour #TravelDiaries #BeachLife #Wanderlust #NatureLover

[SiliconFlow · deepseek/deepseek-chat-v3-0324]
  tokens → prompt: 114 | completion: 67 | reasoning: 0 | cost: $0.00009550
--- S2: productivity and work from home ---
Early mornings and fresh starts. The coffee is warm, my notes are open, and this little plant keeps me company while I tackle the to-do list—one keystroke at a time.
#WorkFromHome #ProductivityMode #DeskSetup #MorningRoutine #WFHLife #CreativeSpace #StayInspired

[DeepInfra · deepseek/deepseek-chat-v3-0324]
 

In [30]:
print("=== ROUND 3: Generate + Self-evaluate ===\n")

def generate_and_rank(scenario: dict, tones: list, model: str = "deepseek/deepseek-chat-v3-0324") -> dict:
    # Generate one caption per tone
    candidates = []
    for tone in tones:
        r = generate_caption(scenario, tone=tone)
        candidates.append({"tone": tone, "caption": r["full"]})

    # Ask the model to pick the best one and explain why
    candidate_text = "\n\n".join([
        f"Option {i+1} ({c['tone']}):\n{c['caption']}"
        for i, c in enumerate(candidates)
    ])

    eval_result = query_model(
        model=model,
        prompt=f"""Here are {len(candidates)} Instagram captions for this image:
{scenario['description']}

{candidate_text}

Which caption would perform best on Instagram and why?
Reply with: BEST: Option [N] - [one sentence reason]""",
        system="You are a social media strategist with expertise in Instagram engagement.",
        max_tokens=150,
    )

    return {
        "scenario": scenario["id"],
        "candidates": candidates,
        "verdict": extract_answer(eval_result),
        "eval_cost": eval_result["usage"].get("cost", 0),
    }

ramen = scenarios[2]
ranked = generate_and_rank(ramen, tones=["inspirational", "witty", "minimal", "storytelling"])

print(f"Scenario: {ramen['description']}\n")
for i, c in enumerate(ranked["candidates"]):
    print(f"Option {i+1} ({c['tone']}):")
    print(c["caption"])
    print()

print("=== MODEL VERDICT ===")
print(ranked["verdict"])

=== ROUND 3: Generate + Self-evaluate ===

[DeepInfra · deepseek/deepseek-chat-v3-0324]
  tokens → prompt: 107 | completion: 75 | reasoning: 0 | cost: $0.00008982
[SiliconFlow · deepseek/deepseek-chat-v3-0324]
  tokens → prompt: 111 | completion: 54 | reasoning: 0 | cost: $0.00008175
[SiliconFlow · deepseek/deepseek-chat-v3-0324]
  tokens → prompt: 109 | completion: 38 | reasoning: 0 | cost: $0.00006525
[Novita · deepseek/deepseek-chat-v3-0324]
  tokens → prompt: 108 | completion: 86 | reasoning: 0 | cost: $0.00012548
[Novita · deepseek/deepseek-chat-v3-0324]
  tokens → prompt: 353 | completion: 60 | reasoning: 0 | cost: $0.00016251
Scenario: A bowl of colorful ramen with soft boiled egg, nori and sesame seeds

Option 1 (inspirational):
Every bite is a burst of flavor and color—like sunshine in a bowl. The silky egg, crispy nori, and nutty sesame seeds dance together in perfect harmony. Who else feels instantly happier with a warm bowl of ramen?  

#RamenLove #FoodieAdventures #NoodleH

In [ ]:
# Task 6 observations
observations = {
    "best_performer": "storytelling — sensory specificity drives engagement",
    "weakest": "witty — puns are hard to enforce via prompt; output is formulaic",
    "underrated": "minimal — wins on aesthetic accounts, loses on general foodie",
    "key_lesson": "self-evaluation needs audience context to rank correctly",
    "llm_as_judge": "model verdict matched human intuition — same pattern as RAGAS"
}

import json
print(json.dumps(observations, indent=2))

In [31]:
print("=== TASK 6 SUMMARY ===\n")
print(f"{'Scenario':<6} {'Tone':<16} {'Tokens':<10} {'Cost'}")
print("-" * 48)

all_results = tone_results + scenario_results
for r in all_results:
    print(f"{r['id']:<6} {r['tone']:<16} {r['tokens']:<10} ${r['cost']:.8f}")

print("-" * 48)
print(f"\nRound 3 ranking call cost: ${ranked['eval_cost']:.8f}")

=== TASK 6 SUMMARY ===

Scenario Tone             Tokens     Cost
------------------------------------------------
s1     inspirational    77         $0.00011432
s1     witty            38         $0.00006500
s1     minimal          28         $0.00005450
s1     storytelling     73         $0.00011011
s1     storytelling     77         $0.00011823
s2     storytelling     67         $0.00009550
s3     storytelling     80         $0.00009456
------------------------------------------------

Round 3 ranking call cost: $0.00016251


## Phase 4 · Model Comparison Harness

Which DeepSeek model gives the best output per dollar across task types?
We run the same prompt through V3, R1, and R1-0528 and score each output.

In [32]:
import time
import json

MODELS = {
    "v3": "deepseek/deepseek-chat-v3-0324",
    "r1": "deepseek/deepseek-r1",
    "r1_0528": "deepseek/deepseek-r1-0528",
}

def score_output(
    task_type: str,
    prompt: str,
    output: str,
    model: str = "deepseek/deepseek-chat-v3-0324",
) -> float:
    """
    Ask the model to score an output 1-10 for a given task type.
    Returns float score.
    """
    criteria = {
        "email": "professionalism, empathy, clarity, and actionability",
        "code": "correctness, readability, and edge case handling",
        "summarization": "accuracy, conciseness, and information retention",
        "caption": "engagement, creativity, and audience fit",
    }

    result = query_model(
        model=model,
        prompt=f"""Score this {task_type} output from 1-10 based on {criteria[task_type]}.

Original prompt: {prompt}

Output to score:
{output}

Reply with ONLY a number between 1 and 10. No explanation.""",
        max_tokens=10,
        verbose=False,
    )

    try:
        return float(extract_answer(result).strip())
    except ValueError:
        return 0.0

In [33]:
def run_benchmark(
    task_type: str,
    prompt: str,
    system: str = None,
    max_tokens: int = 500,
) -> list:
    """
    Run the same prompt through all three models.
    Returns list of result dicts with output, tokens, cost, latency, score.
    """
    results = []

    for model_key, model_id in MODELS.items():
        print(f"  Running {model_key}...", end=" ")

        start = time.time()
        result = query_model(
            model=model_id,
            prompt=prompt,
            system=system,
            max_tokens=max_tokens,
            include_reasoning=True,
            verbose=False,
        )
        latency = round(time.time() - start, 2)

        output = extract_answer(result)
        score = score_output(task_type, prompt, output)

        usage = result["usage"]
        r_tokens = usage.get("completion_tokens_details", {}).get("reasoning_tokens", 0)

        results.append({
            "model": model_key,
            "model_id": model_id,
            "output": output,
            "score": score,
            "latency": latency,
            "prompt_tokens": usage.get("prompt_tokens", 0),
            "completion_tokens": usage.get("completion_tokens", 0),
            "reasoning_tokens": r_tokens,
            "cost": usage.get("cost", 0),
            "provider": result.get("provider"),
        })
        print(f"score={score} | {latency}s | ${usage.get('cost', 0):.8f}")

    return results

In [34]:
print("=== PHASE 4: MODEL COMPARISON HARNESS ===\n")

benchmarks = {}

if "paper_excerpt" not in globals():
    paper_path = project_root / "data" / "paper_excerpt.txt"
    paper_excerpt = paper_path.read_text(encoding="utf-8")

# Email task
print("[ Email - angry customer review ]")
benchmarks["email"] = run_benchmark(
    task_type="email",
    prompt="Write a customer support email reply to this review: 'The product broke after one day. Complete waste of money.'",
    system="You are a professional customer support specialist. Write empathetic, actionable email replies. No preamble. Sign off as The Support Team.",
    max_tokens=300,
)

print("\n[ Code - flatten nested list ]")
benchmarks["code"] = run_benchmark(
    task_type="code",
    prompt="Write a Python function called flatten that takes a nested list of arbitrary depth and returns a flat list.",
    system="You are an expert Python engineer. Return only the function with a docstring. No usage examples.",
    max_tokens=500,
)

print("\n[ Summarization - transformer paper ]")
benchmarks["summarization"] = run_benchmark(
    task_type="summarization",
    prompt=f"Summarize this in 4 bullet points for a technical audience:\n\n{paper_excerpt}",
    max_tokens=300,
)

print("\n[ Caption - ramen bowl ]")
benchmarks["caption"] = run_benchmark(
    task_type="caption",
    prompt="Write an Instagram caption for: A bowl of colorful ramen with soft boiled egg, nori and sesame seeds",
    system="You are a professional Instagram content creator. Storytelling tone. Add 5 hashtags on a new line. No preamble.",
    max_tokens=200,
)

=== PHASE 4: MODEL COMPARISON HARNESS ===

[ Email - angry customer review ]
  Running v3... score=8.0 | 4.24s | $0.00011856
  Running r1... score=8.0 | 7.91s | $0.00078570
  Running r1_0528... score=9.0 | 28.47s | $0.00109906

[ Code - flatten nested list ]
  Running v3... score=9.0 | 8.57s | $0.00021775
  Running r1... score=8.0 | 5.5s | $0.00303682
  Running r1_0528... score=7.0 | 15.54s | $0.00109510

[ Summarization - transformer paper ]
  Running v3... score=9.0 | 6.42s | $0.00032031
  Running r1... score=8.0 | 10.4s | $0.00102510
  Running r1_0528... score=8.0 | 11.8s | $0.00078490

[ Caption - ramen bowl ]
  Running v3... score=8.0 | 3.17s | $0.00010086
  Running r1... score=7.0 | 3.23s | $0.00126522
  Running r1_0528... score=9.0 | 14.96s | $0.00054266


In [35]:
print("\n=== COMPARISON TABLE ===\n")

header = f"{'Task':<16} {'Model':<10} {'Score':<8} {'Latency':<10} {'C-Tokens':<10} {'R-Tokens':<10} {'Cost':<14} {'Provider'}"
print(header)
print("-" * 90)

all_rows = []
for task, results in benchmarks.items():
    for r in results:
        row = {
            "task": task,
            "model": r["model"],
            "score": r["score"],
            "latency": r["latency"],
            "c_tokens": r["completion_tokens"],
            "r_tokens": r["reasoning_tokens"],
            "cost": r["cost"],
            "provider": r["provider"],
        }
        all_rows.append(row)
        print(
            f"{task:<16} {r['model']:<10} {r['score']:<8} "
            f"{r['latency']:<10} {r['completion_tokens']:<10} "
            f"{r['reasoning_tokens']:<10} ${r['cost']:<13.8f} {r['provider']}"
        )
    print()


=== COMPARISON TABLE ===

Task             Model      Score    Latency    C-Tokens   R-Tokens   Cost           Provider
------------------------------------------------------------------------------------------
email            v3         8.0      4.24       120        0          $0.00011856    DeepInfra
email            r1         8.0      7.91       300        386        $0.00078570    Novita
email            r1_0528    9.0      28.47      492        345        $0.00109906    SiliconFlow

code             v3         9.0      8.57       207        0          $0.00021775    SiliconFlow
code             r1         8.0      5.5        500        557        $0.00303682    Azure
code             r1_0528    7.0      15.54      500        591        $0.00109510    DeepInfra

summarization    v3         9.0      6.42       181        0          $0.00032031    GMICloud
summarization    r1         8.0      10.4       300        382        $0.00102510    Novita
summarization    r1_0528    8.0  

In [36]:
try:
    import matplotlib.pyplot as plt
    import numpy as np
except ImportError:
    plt = None
    np = None

if plt is None or np is None:
    print("Skipping chart: matplotlib/numpy not available in this environment.")
else:
    tasks = list(benchmarks.keys())
    model_keys = list(MODELS.keys())
    colors = {"v3": "#3B8BD4", "r1": "#E8593C", "r1_0528": "#1D9E75"}

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Chart 1: Score by task and model
    x = np.arange(len(tasks))
    width = 0.25

    ax1 = axes[0]
    for i, model_key in enumerate(model_keys):
        scores = [
            next(r["score"] for r in benchmarks[t] if r["model"] == model_key)
            for t in tasks
        ]
        ax1.bar(x + i * width, scores, width, label=model_key,
                color=colors[model_key], alpha=0.85)

    ax1.set_xlabel("Task")
    ax1.set_ylabel("Quality score (1-10)")
    ax1.set_title("Output quality by model and task")
    ax1.set_xticks(x + width)
    ax1.set_xticklabels(tasks, rotation=15)
    ax1.set_ylim(0, 11)
    ax1.legend()
    ax1.grid(axis="y", alpha=0.3)

    # Chart 2: Cost by task and model
    ax2 = axes[1]
    for i, model_key in enumerate(model_keys):
        costs = [
            next(r["cost"] for r in benchmarks[t] if r["model"] == model_key)
            for t in tasks
        ]
        ax2.bar(x + i * width, costs, width, label=model_key,
                color=colors[model_key], alpha=0.85)

    ax2.set_xlabel("Task")
    ax2.set_ylabel("Cost ($)")
    ax2.set_title("Cost by model and task")
    ax2.set_xticks(x + width)
    ax2.set_xticklabels(tasks, rotation=15)
    ax2.legend()
    ax2.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig("../data/phase4_comparison.png", dpi=150)
    plt.show()
    print("Chart saved to data/")

Skipping chart: matplotlib/numpy not available in this environment.


In [37]:
print("=== VERDICT ===\n")

for task, results in benchmarks.items():
    best_score = max(results, key=lambda r: r["score"])
    best_value = max(results, key=lambda r: r["score"] / r["cost"] if r["cost"] > 0 else 0)
    cheapest = min(results, key=lambda r: r["cost"])

    print(f"{task.upper()}")
    print(f"  Best quality:  {best_score['model']} "
          f"(score {best_score['score']}, ${best_score['cost']:.8f})")
    print(f"  Best value:    {best_value['model']} "
          f"(score/cost ratio)")
    print(f"  Cheapest:      {cheapest['model']} "
          f"(${cheapest['cost']:.8f})")
    print()

=== VERDICT ===

EMAIL
  Best quality:  r1_0528 (score 9.0, $0.00109906)
  Best value:    v3 (score/cost ratio)
  Cheapest:      v3 ($0.00011856)

CODE
  Best quality:  v3 (score 9.0, $0.00021775)
  Best value:    v3 (score/cost ratio)
  Cheapest:      v3 ($0.00021775)

SUMMARIZATION
  Best quality:  v3 (score 9.0, $0.00032031)
  Best value:    v3 (score/cost ratio)
  Cheapest:      v3 ($0.00032031)

CAPTION
  Best quality:  r1_0528 (score 9.0, $0.00054266)
  Best value:    v3 (score/cost ratio)
  Cheapest:      v3 ($0.00010086)



In [ ]:
conclusions = {
    "headline": "V3 wins on value across all task types at 5-10x lower cost",

    "findings": {
        "email": {
            "recommendation": "V3 for production volume, R1-0528 if quality is critical",
            "insight": "R1-0528 scored 1pt higher but took 28s vs 4s - unacceptable for real-time support",
            "cost_ratio": f"{0.00109906 / 0.00011856:.0f}x more expensive for R1-0528 vs V3"
        },
        "code": {
            "recommendation": "Retest R1 with max_tokens=2000 - both R1 models hit the 500 token cap",
            "insight": "Token cap truncation likely caused R1 underperformance, not model capability",
            "cost_ratio": f"{0.00303682 / 0.00021775:.0f}x more expensive for R1 vs V3"
        },
        "summarization": {
            "recommendation": "V3 - clear winner on both quality and cost",
            "insight": "Summarization is pattern-matching and compression; reasoning adds no value",
            "cost_ratio": f"{0.00102510 / 0.00032031:.0f}x more expensive for R1 vs V3"
        },
        "caption": {
            "recommendation": "V3 for volume creative work; R1-0528 for hero content",
            "insight": "Creative tasks don't benefit from chain-of-thought reasoning",
            "cost_ratio": f"{0.00126522 / 0.00010086:.0f}x more expensive for R1 vs V3"
        },
    },

    "meta_findings": [
        "LLM-as-judge scoring has ~1pt noise - don't over-interpret single-point differences",
        "Token cap is a confound - reasoning models need 2000+ tokens to show true capability",
        "Latency matters as much as quality for production use cases",
        "OpenRouter routed across 6 providers in one benchmark - gateway abstraction works",
    ],

    "when_to_use_r1": [
        "Complex multi-step reasoning (math, logic puzzles, architectural decisions)",
        "One-shot tasks where latency is acceptable and quality is critical",
        "Tasks where the reasoning trace itself has value (explainability, auditing)",
    ]
}

import json
print(json.dumps(conclusions, indent=2))

In [ ]:
print("=== BONUS: Code retest with proper token budget ===\n")

code_retest = run_benchmark(
    task_type="code",
    prompt="Write a Python function called flatten that takes a nested list of arbitrary depth and returns a flat list.",
    system="You are an expert Python engineer. Return only the function with a docstring. No usage examples.",
    max_tokens=2000,
 )

print(f"\n{'Model':<10} {'Score':<8} {'C-Tokens':<10} {'R-Tokens':<10} {'Cost'}")
print("-" * 50)
for r in code_retest:
    print(f"{r['model']:<10} {r['score']:<8} {r['completion_tokens']:<10} "
          f"{r['reasoning_tokens']:<10} ${r['cost']:.8f}")